# 02 · Train matched classifier heads

MFCC and HuBERT use the same speaker splits, loss weighting, architecture, optimizer, and selection metric. Only the fixed feature vectors differ. This notebook uses the RTX 4060 for both MLP training runs and never inspects test labels or scores.


In [1]:
from pathlib import Path
import json, random
import joblib
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import f1_score
from sklearn.preprocessing import StandardScaler
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

ROOT = Path.cwd()
FEATURES = ROOT / 'features'
RUNS = ROOT / 'runs'
RUNS.mkdir(exist_ok=True)
manifest = pd.read_csv(ROOT / 'data' / 'manifest.csv').sort_values('utterance_id').reset_index(drop=True)
CLASSES = ['Hindi','Kannada','Malayalam','Tamil','Telugu']
CLASS_TO_INDEX = {name:i for i,name in enumerate(CLASSES)}
DEVICE = torch.device('cuda:0')
assert torch.cuda.is_available(), 'CUDA is required for the final training run.'
assert '4060' in torch.cuda.get_device_name(DEVICE)
print('Training on', torch.cuda.get_device_name(DEVICE))


Training on NVIDIA GeForce RTX 4060 Laptop GPU


## 1. Features and speaker-balanced training weights

Each language receives equal total weight, and each speaker within that language receives equal weight. Standardization uses only training utterances.


In [2]:
def load_features(name):
    package = np.load(FEATURES / f'{name}.npz')
    ids = manifest.utterance_id.astype(str).to_numpy()
    assert np.array_equal(package['utterance_id'].astype(str), ids)
    X = package['X'].astype('float32')
    assert len(X) == len(manifest) and np.isfinite(X).all()
    return X

labels = manifest.language.map(CLASS_TO_INDEX).to_numpy('int64')
train_mask = manifest.split.eq('train').to_numpy()
val_mask = manifest.split.eq('val').to_numpy()
assert train_mask.any() and val_mask.any()
train = manifest.loc[train_mask]
class_speakers = train.groupby('language').speaker_id.nunique().to_dict()
speaker_utterances = train.groupby('speaker_id').size().to_dict()
weights = np.array([1 / (class_speakers[row.language] * speaker_utterances[row.speaker_id])
                    for row in train.itertuples(index=False)], dtype='float32')
weights /= weights.mean()
print('Train / validation:', train_mask.sum(), val_mask.sum())
print('Training speakers by language:', class_speakers)


Train / validation: 6337 1638
Training speakers by language: {'Hindi': 47, 'Kannada': 23, 'Malayalam': 29, 'Tamil': 28, 'Telugu': 24}


## 2. Shared network and validation scoring


In [3]:
class Classifier(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.layers = nn.Sequential(nn.Linear(input_dim,128), nn.GELU(),
                                    nn.Dropout(0.2), nn.Linear(128,5))
    def forward(self, features):
        return self.layers(features)

def speaker_macro_f1(y_true, probabilities, speakers):
    frame = pd.DataFrame(probabilities, columns=CLASSES)
    frame['speaker_id'] = np.asarray(speakers)
    frame['truth'] = y_true
    pooled = frame.groupby('speaker_id', sort=True)[CLASSES].mean()
    truth = frame.groupby('speaker_id', sort=True).truth.first().loc[pooled.index].to_numpy()
    return f1_score(truth, pooled.to_numpy().argmax(axis=1),
                    labels=np.arange(5), average='macro', zero_division=0)


In [4]:
def predict_probabilities(model, loader):
    model.eval()
    output = []
    with torch.inference_mode():
        for (features,) in loader:
            logits = model(features.to(DEVICE, non_blocking=True))
            output.append(logits.softmax(dim=1).cpu().numpy())
    return np.concatenate(output)

def train_epoch(model, loader, optimizer):
    model.train()
    total, weight_total = 0.0, 0.0
    for features, target, sample_weight in loader:
        features = features.to(DEVICE, non_blocking=True)
        target = target.to(DEVICE, non_blocking=True)
        sample_weight = sample_weight.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        losses = nn.functional.cross_entropy(model(features), target, reduction='none')
        loss = (losses * sample_weight).sum() / sample_weight.sum()
        loss.backward()
        optimizer.step()
        total += float((losses.detach() * sample_weight).sum())
        weight_total += float(sample_weight.sum())
    return total / weight_total


## 3. Train each representation

The best checkpoint is selected by validation speaker macro-F1. Ties retain the earlier epoch. Test examples are absent from all fitted scalers and training loaders.


In [5]:
def train_model(name, max_epochs=100, patience=10, batch_size=64, seed=42):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    X = load_features(name)
    scaler = StandardScaler().fit(X[train_mask])
    scaled = scaler.transform(X).astype('float32')
    joblib.dump(scaler, RUNS / f'{name}_scaler.joblib')
    train_set = TensorDataset(torch.from_numpy(scaled[train_mask]),
                              torch.from_numpy(labels[train_mask]), torch.from_numpy(weights))
    val_set = TensorDataset(torch.from_numpy(scaled[val_mask]))
    train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True, pin_memory=True)
    val_loader = DataLoader(val_set, batch_size=256, pin_memory=True)
    model = Classifier(scaled.shape[1]).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    history, best, stale = [], -1.0, 0
    for epoch in range(1, max_epochs + 1):
        loss = train_epoch(model, train_loader, optimizer)
        probabilities = predict_probabilities(model, val_loader)
        score = speaker_macro_f1(labels[val_mask], probabilities,
                                 manifest.loc[val_mask, 'speaker_id'].to_numpy())
        history.append({'epoch':epoch, 'train_loss':loss, 'val_speaker_macro_f1':score})
        if score > best + 1e-8:
            best, stale = score, 0
            torch.save({'state_dict':model.state_dict(), 'input_dim':scaled.shape[1],
                        'classes':CLASSES, 'epoch':epoch, 'val_speaker_macro_f1':score,
                        'seed':seed}, RUNS / f'{name}_best.pt')
        else:
            stale += 1
        if epoch == 1 or epoch % 10 == 0 or stale == patience:
            print(f'{name} epoch {epoch:03d}: loss={loss:.4f}, val speaker macro-F1={score:.4f}')
        if stale >= patience:
            break
    pd.DataFrame(history).to_csv(RUNS / f'{name}_history.csv', index=False)
    print(f'{name}: best validation speaker macro-F1={best:.4f}')


In [6]:
for representation in ('mfcc','hubert'):
    train_model(representation)

for representation in ('mfcc','hubert'):
    checkpoint = torch.load(RUNS / f'{representation}_best.pt', map_location='cpu', weights_only=False)
    scaler = joblib.load(RUNS / f'{representation}_scaler.joblib')
    assert checkpoint['input_dim'] == scaler.n_features_in_
    restored = Classifier(checkpoint['input_dim'])
    restored.load_state_dict(checkpoint['state_dict'])
    print(representation, 'checkpoint reload verified at epoch', checkpoint['epoch'])


mfcc epoch 001: loss=1.2285, val speaker macro-F1=0.5641


mfcc epoch 010: loss=0.1660, val speaker macro-F1=0.7580


mfcc epoch 014: loss=0.1153, val speaker macro-F1=0.7580
mfcc: best validation speaker macro-F1=0.8027


hubert epoch 001: loss=0.3821, val speaker macro-F1=0.8741


hubert epoch 010: loss=0.0563, val speaker macro-F1=0.9253


hubert epoch 019: loss=0.0011, val speaker macro-F1=0.8959
hubert: best validation speaker macro-F1=0.9253
mfcc checkpoint reload verified at epoch 4
hubert checkpoint reload verified at epoch 9


The next notebook evaluates the fixed test speakers once and creates the comparison artifacts.
